# 2. Dual-Model Pipeline Prototyping

In this notebook, we prototyping the final, production-ready ML architecture:
1. **Separate Rank Scales**: We split the dataset into IITs (JEE Advanced) and NIT/IIITs (JEE Mains).
2. **Prevent Data Leakage**: We completely drop `Opening Rank` so the model must genuinely learn cutoffs based purely on category, seat type, and round demand.
3. **Independent Encoders & Models**: We train, evaluate, and verify separate pipelines.

In [1]:
import sys
import os
import time

# Add the 'src' directory to the Python path
sys.path.append(os.path.abspath('..'))

from src.data_loader import DataLoader
from src.features import split_and_preprocess

## Step 1: Load, Clean, & Split
We load the data, apply basic cleaning, drop the leaked variable, and isolate our 8 matrices for training.

In [2]:
DATA_PATH = '../data/raw/merged_jee_cutoff_2018_2025.csv'

loader = DataLoader(DATA_PATH)
raw_df = loader.load_data()
clean_df = loader.basic_clean()

print("\nStarting Preprocessing and Dual Data Split...")
(
    X_train_iit, X_test_iit, y_train_iit, y_test_iit, 
    X_train_nit, X_test_nit, y_train_nit, y_test_nit
) = split_and_preprocess(clean_df)

Loading data from ../data/raw/merged_jee_cutoff_2018_2025.csv...
Data successfully loaded. Shape: (Rows, Columns) (433993, 9)

Performing basic cleaning...
  -> Dropped 1469 rows due to missing Ranks.
  -> Standardized 'Gender' column values.

Starting Preprocessing and Dual Data Split...
----------------------------------------
FEATURE PREPROCESSING & DATA SPLIT
----------------------------------------
Dropped 'Opening Rank' to prevent data leakage.
Data split complete:
  -> IIT Dataset shape: (102876, 8)
  -> NIT+ Dataset shape: (329648, 8)

Encoding IIT Dataset...
Saved IIT Label Encoders to: d:\DEV\AIML\PROJECTS\Predictiv-Jee\src\..\models\iit_encoders.pkl

Encoding NIT/IIIT/GFTI Dataset...
Saved NIT Label Encoders to: d:\DEV\AIML\PROJECTS\Predictiv-Jee\src\..\models\nit_encoders.pkl

Performing Train-Test Splits...
IIT Train shape: (82300, 7), Test shape: (20576, 7)
NIT Train shape: (263718, 7), Test shape: (65930, 7)


## Step 2: Train & Evaluate IIT Model
We initialize a Random Forest with 100 trees specifically configured to predict JEE Advanced ranks for IITs.

In [3]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

print("Initializing and training IIT Random Forest...")
model_iit = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

start = time.time()
model_iit.fit(X_train_iit, y_train_iit)
end = time.time()
print(f"[SUCCESS] IIT Training Complete in {end-start:.2f} seconds.")

# Evaluation
y_pred_iit = model_iit.predict(X_test_iit)
mae_iit = mean_absolute_error(y_test_iit, y_pred_iit)
r2_iit = r2_score(y_test_iit, y_pred_iit)

print("\n" + "-"*40)
print("IIT MODEL EVALUATION (Unseen Test Set)")
print("-"*40)
print(f"Mean Absolute Error (MAE): {mae_iit:.2f} ranks")
print(f"R-squared (R²): {r2_iit:.4f}")

Initializing and training IIT Random Forest...
[SUCCESS] IIT Training Complete in 2.67 seconds.

----------------------------------------
IIT MODEL EVALUATION (Unseen Test Set)
----------------------------------------
Mean Absolute Error (MAE): 76.58 ranks
R-squared (R²): 0.9976


## Step 3: Train & Evaluate NIT Model
Next, we train the much larger NIT/IIIT/GFTI model which deals with JEE Mains scale ranks.

In [4]:
print("Initializing and training NIT Random Forest...")
model_nit = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

start = time.time()
model_nit.fit(X_train_nit, y_train_nit)
end = time.time()
print(f"[SUCCESS] NIT Training Complete in {end-start:.2f} seconds.")

# Evaluation
y_pred_nit = model_nit.predict(X_test_nit)
mae_nit = mean_absolute_error(y_test_nit, y_pred_nit)
r2_nit = r2_score(y_test_nit, y_pred_nit)

print("\n" + "-"*40)
print("NIT/IIIT/GFTI MODEL EVALUATION (Unseen Test Set)")
print("-"*40)
print(f"Mean Absolute Error (MAE): {mae_nit:.2f} ranks")
print(f"R-squared (R²): {r2_nit:.4f}")

Initializing and training NIT Random Forest...
[SUCCESS] NIT Training Complete in 20.73 seconds.

----------------------------------------
NIT/IIIT/GFTI MODEL EVALUATION (Unseen Test Set)
----------------------------------------
Mean Absolute Error (MAE): 1344.69 ranks
R-squared (R²): 0.9616


## Step 4: Local Simulation test
Let's import our new `predict.py` module and run a quick prediction inside this notebook to make sure the entire backend wiring matches!

In [5]:
from src.predict import predict_closing_rank

# 1. Simulated NIT Input
test_nit = {
    'Institute': 'National Institute of Technology Durgapur',
    'Academic Program Name': 'Computer Science and Engineering (4 Years, Bachelor of Technology)',
    'Quota': 'OS',
    'Seat Type': 'OPEN',
    'Gender': 'Gender-Neutral',
    'Round': 6,
    'Year': 2025
}

# 2. Simulated IIT Input
test_iit = {
    'Institute': 'Indian Institute of Technology Bombay',
    'Academic Program Name': 'Computer Science and Engineering (4 Years, Bachelor of Technology)',
    'Quota': 'AI',
    'Seat Type': 'OPEN',
    'Gender': 'Gender-Neutral',
    'Round': 6,
    'Year': 2025
}

print(f"NIT Prediction -> Predicted Closing Rank: {predict_closing_rank(test_nit)}")
print(f"IIT Prediction -> Predicted Closing Rank: {predict_closing_rank(test_iit)}")

d:\DEV\AIML\PROJECTS\Predictiv-Jee\.venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
d:\DEV\AIML\PROJECTS\Predictiv-Jee\.venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator RandomForestRegressor from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


NIT Prediction -> Predicted Closing Rank: 9047
IIT Prediction -> Predicted Closing Rank: 66
